# 03 - Volatility Modeling & Forecasting Pipeline

This notebook acts as an execution and inspection runner for the hybrid volatility forecasting pipeline (AR(1)-GARCH(1,1) + Sentiment-LSTM).

When running on Kaggle, this notebook will:
- Read keys (`GITHUB_TOKEN`, etc.) from Kaggle Secrets.
- Clone or pull the active branch of the GitHub repo into `/kaggle/working`.
- Materialize credentials in `.dvc/credentials/key.json` and `.dvc/config.local` for service account access.
- Pull DVC tracking targets (like raw news files, prices, and pre-computed sentiment scores).
- Install dependencies and run the volatility pipeline stages against the checked-out workspace.

## 0 - Bootstrap

In [ ]:
import base64
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

STAGED_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
IS_KAGGLE = Path('/kaggle').exists()

def run_command(*args: str, cwd: str | Path | None = None, display: str | None = None) -> None:
    command = [str(arg) for arg in args]
    print('$', display or ' '.join(command))
    subprocess.run(command, check=True, cwd=cwd)

def load_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'))

def load_kaggle_secret(name: str) -> str | None:
    if not IS_KAGGLE:
        return os.getenv(name)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)

def parse_json_secret(value: str | None) -> dict | None:
    if not value:
        return None
    candidates = [value]
    try:
        decoded = base64.b64decode(value).decode('utf-8')
        candidates.append(decoded)
    except Exception:
        pass
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            continue
    return None

def write_service_account_key(secret_name: str, repo_root: Path) -> Path | None:
    secret_payload = parse_json_secret(load_kaggle_secret(secret_name))
    if not secret_payload:
        return None
    key_path = repo_root / '.dvc' / 'credentials' / 'key.json'
    key_path.parent.mkdir(parents=True, exist_ok=True)
    key_path.write_text(json.dumps(secret_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return key_path

def configure_dvc_service_account(repo_root: Path, key_path: Path, remote_name: str = 'gdrive') -> Path:
    config_local_path = repo_root / '.dvc' / 'config.local'
    rel_key_path = key_path.relative_to(repo_root / '.dvc')
    config_lines = [
        f'[remote "{remote_name}"]',
        '    gdrive_use_service_account = true',
        f'    gdrive_service_account_json_file_path = {rel_key_path.as_posix()}'
    ]
    config_local_path.write_text('\n'.join(config_lines) + '\n', encoding='utf-8')
    return config_local_path

def github_auth_url(repo_url: str, token: str | None) -> str:
    if not token or not repo_url.startswith('https://github.com/'):
        return repo_url
    return repo_url.replace('https://', f'https://{token}@', 1)

print('Running on Kaggle:', IS_KAGGLE)
print('Staged root     :', STAGED_ROOT)


## 1 - Runtime Configuration

In [ ]:
REPO_GIT_URL = 'https://github.com/tlong-ds/news-sentiment-analysis.git'
REPO_BRANCH = 'main'
REPO_DIR_NAME = 'news-sentiment-analysis'
PULL_REPO_ON_KAGGLE = IS_KAGGLE
INSTALL_PROJECT_REQUIREMENTS = IS_KAGGLE
LOAD_KAGGLE_SECRETS = IS_KAGGLE
GDRIVE_SERVICE_ACCOUNT_SECRET = 'GDRIVE_SERVICE_ACCOUNT_JSON'
SETUP_DVC_SERVICE_ACCOUNT = IS_KAGGLE
DVC_PULL_ON_KAGGLE = IS_KAGGLE
DVC_PULL_REMOTE = 'gdrive'

# We pull raw data and existing sentiment parquets to run volatility modeling
DVC_PULL_TARGETS = [
    'data/raw.dvc',
    'pipelines/sentiment/dvc.yaml'
]


## 2 - Kaggle Secrets, Repo Pull, and DVC Bootstrap

In [ ]:
if LOAD_KAGGLE_SECRETS:
    for secret_name in ['GITHUB_TOKEN', 'GEMINI_API_KEY']:
        secret_value = load_kaggle_secret(secret_name)
        if secret_value and secret_name not in os.environ:
            os.environ[secret_name] = secret_value

if IS_KAGGLE and PULL_REPO_ON_KAGGLE:
    kaggle_repo_root = Path('/kaggle/working') / REPO_DIR_NAME
    auth_repo_url = github_auth_url(REPO_GIT_URL, os.getenv('GITHUB_TOKEN'))

    if kaggle_repo_root.exists() and (kaggle_repo_root / '.git').exists():
        run_command('git', 'fetch', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'checkout', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'pull', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
    else:
        run_command(
            'git', 'clone', '--branch', REPO_BRANCH, auth_repo_url, str(kaggle_repo_root),
            cwd='/kaggle/working'
        )
    ROOT = kaggle_repo_root
else:
    ROOT = STAGED_ROOT

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
importlib.invalidate_caches()

if INSTALL_PROJECT_REQUIREMENTS:
    run_command(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', cwd=ROOT)
else:
    print('Skipping pip install.')

service_account_key_path = None
dvc_config_local_path = None
if SETUP_DVC_SERVICE_ACCOUNT:
    service_account_key_path = write_service_account_key(GDRIVE_SERVICE_ACCOUNT_SECRET, ROOT)
    if service_account_key_path is None:
        print(f'{GDRIVE_SERVICE_ACCOUNT_SECRET} not found or invalid; skipping DVC service-account setup.')
    else:
        dvc_config_local_path = configure_dvc_service_account(ROOT, service_account_key_path, remote_name=DVC_PULL_REMOTE)
        os.environ.setdefault('DVC_SITE_CACHE_DIR', str(ROOT / '.dvc' / 'site-cache'))
        if DVC_PULL_ON_KAGGLE:
            run_command('dvc', 'pull', '--force', *DVC_PULL_TARGETS, cwd=ROOT)
else:
    print('SETUP_DVC_SERVICE_ACCOUNT is False; skipping DVC auth bootstrap.')

def run_module(module: str, *args: str) -> None:
    run_command(sys.executable, '-m', module, *args, cwd=ROOT)

print('ROOT                :', ROOT)


## 3 - Volatility Pipeline Parameters

Here we configure the dataset paths and model parameters. These align with the central configurations in `params.yaml`.

In [ ]:
PRICES_FILE = ROOT / 'data' / 'raw' / 'prices_VN.csv'
DAILY_NEWS_FILE = ROOT / 'data' / 'interim' / 'daily_news_prices.parquet'
SENTIMENT_FILE = ROOT / 'data' / 'sentiment' / 'article_sentiment_scores.parquet'
ARTICLES_CLEAN_FILE = ROOT / 'data' / 'interim' / 'articles_clean.parquet'
MODELING_READY_FILE = ROOT / 'data' / 'interim' / 'modeling_ready.parquet'

TARGET_TYPE = 'parkinson'
SENTIMENT_THRESHOLD = 0.05
SEQUENCE_LENGTH = 15
TRAIN_END = '2021-12-31'
VAL_END = '2021-12-31'
EPOCHS = 30
BATCH_SIZE = 32

HYBRID_OUTPUT = ROOT / 'data' / 'interim' / 'hybrid_experiment_summary.json'
ROBUSTNESS_OUTPUT = ROOT / 'data' / 'interim' / 'robustness_experiment_summary.json'

for path in [MODELING_READY_FILE.parent, HYBRID_OUTPUT.parent]:
    path.mkdir(parents=True, exist_ok=True)


## 4 - Stage 1: Build the Model Frame

Construct the model frame aligning return volatility (Parkinson) with daily aggregated article sentiments filtered on thresholds and category filters.

In [ ]:
run_module(
    'src.modeling.prepare_model_frame',
    '--prices', str(PRICES_FILE),
    '--daily-news', str(DAILY_NEWS_FILE),
    '--sentiment', str(SENTIMENT_FILE),
    '--articles-clean', str(ARTICLES_CLEAN_FILE),
    '--output-file', str(MODELING_READY_FILE),
    '--target-type', TARGET_TYPE,
    '--sentiment-threshold', str(SENTIMENT_THRESHOLD)
)

df = pd.read_parquet(MODELING_READY_FILE)
print("Modeling Frame Shape:", df.shape)
df.head(3)


## 5 - Stage 2: Fit Econometrics Baseline & Train LSTM

Fits the AR(1)-GARCH(1,1) model on the training period, builds rolling sequences of features (using GARCH conditional variance instead of volatility), and trains the residual-correction LSTM.

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("Device Available:", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

run_module(
    'src.modeling.run_experiment',
    '--prices', str(PRICES_FILE),
    '--model-frame', str(MODELING_READY_FILE),
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--train-end', TRAIN_END,
    '--val-end', VAL_END,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--output', str(HYBRID_OUTPUT)
)

summary = load_json(HYBRID_OUTPUT)
print("\n--- Primary Experiment Complete ---")
print("Baseline RMSE:", summary["baseline_metrics"]["rmse"])
print("Hybrid RMSE  :", summary["hybrid_metrics"]["rmse"])
print("Diebold-Mariano p-value:", summary["diebold_mariano"]["p_value"])


## 6 - Stage 3: Volatility Robustness Checks

Executes extensive robustness checks including alternative model baselines, subsets of macro vs market news categories, and unweighted sentiment metrics.

In [ ]:
run_module(
    'src.modeling.run_robustness',
    '--prices', str(PRICES_FILE),
    '--model-frame', str(MODELING_READY_FILE),
    '--daily-news', str(DAILY_NEWS_FILE),
    '--sentiment', str(SENTIMENT_FILE),
    '--articles-clean', str(ARTICLES_CLEAN_FILE),
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--train-end', TRAIN_END,
    '--val-end', VAL_END,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--output', str(ROBUSTNESS_OUTPUT)
)

robustness = load_json(ROBUSTNESS_OUTPUT)
print("\n--- Robustness Checks Complete ---")
print("Robustness configurations estimated:", list(robustness.keys()))
